In [1]:
import yfinance as yf
import pandas as pd
import numpy as np
import vectorbt as vbt

In [2]:
# Download the data
data = yf.download('SPY', start='2022-01-01', auto_adjust=True)
data.columns = data.columns.droplevel(1)

# Calculate the SMA
data['SMA50'] = data['Close'].rolling(window=50).mean()

# Determine the position state (1 for long, 0 for flat)
data['Position'] = np.where(data['Close'] > data['SMA50'], 1, 0)

# Calculate the signal (1 for buy, -1 for sell, 0 for hold)
data['Signal'] = data['Position'].diff()

# Display the rows where a trade signal occurred
print("Trade Signals (1 = Buy, -1 = Sell):")
print(data[data['Signal'] != 0].head())

[*********************100%***********************]  1 of 1 completed

Trade Signals (1 = Buy, -1 = Sell):
Price            Close        High         Low        Open     Volume  \
Date                                                                    
2022-01-03  453.210388  453.343222  449.548364  451.872697   72668200   
2022-03-18  423.032715  423.356276  416.085595  416.827890  106345500   
2022-04-11  418.655060  423.489489  418.150680  422.642496   89770500   
2022-04-13  421.881134  422.642452  416.675544  416.856362   74070400   
2022-04-14  416.628021  423.232556  416.523323  422.109573   97869500   

Price            SMA50  Position  Signal  
Date                                      
2022-01-03         NaN         0     NaN  
2022-03-18  419.555730         1     1.0  
2022-04-11  419.390385         0    -1.0  
2022-04-13  419.038956         1     1.0  
2022-04-14  418.693625         0    -1.0  


In [3]:
# Create entry signals where Signal is 1 (buy)
entries = data['Signal'] == 1

# Create exit signals where Signal is -1 (sell)
exits = data['Signal'] == -1

In [4]:
# Run the Vectorbt backtest
portfolio = vbt.Portfolio.from_signals(
    close=data['Close'],
    entries=entries,
    exits=exits,
    init_cash=100_000,  # Start with $100,000
    freq='D'           # Use daily frequency for calculations
)

# Print the performance statistics
print("\n--- Backtest Performance ---")
print(portfolio.stats())


--- Backtest Performance ---
Start                                2022-01-03 00:00:00
End                                  2025-10-24 00:00:00
Period                                 957 days 00:00:00
Start Value                                     100000.0
End Value                                   137352.18216
Total Return [%]                               37.352182
Benchmark Return [%]                           49.433909
Max Gross Exposure [%]                             100.0
Total Fees Paid                                      0.0
Max Drawdown [%]                               20.259779
Max Drawdown Duration                  380 days 00:00:00
Total Trades                                          28
Total Closed Trades                                   27
Total Open Trades                                      1
Open Trade PnL                              24735.690541
Win Rate [%]                                   25.925926
Best Trade [%]                                 16.880742
W

In [5]:
# Plot the closing price first to create the base figure
fig = data['Close'].vbt.plot(trace_kwargs=dict(name='Price'))

# Add the SMA50 indicator to the same figure
data['SMA50'].vbt.plot(fig=fig, trace_kwargs=dict(name='SMA50'))

# Add the buy/sell signals from the portfolio to the figure
portfolio.positions.plot(fig=fig)

fig.show()

## Analyze Our Results

* **Total Return vs. Benchmark:** Our strategy returned 37.35%, while simply buying and holding SPY over the same period would have returned 49.43%.

    * **Our strategy, in its current form, did not beat the market.**

* **Win Rate (25.9%):** This is very low. It means we lost on roughly 3 out of every 4 trades. This isn't necessarily a dealbreaker for a trend-following strategy, if the wins are massive and the losses are small. Our "Profit Factor" of 1.48 (total money won / total money lost) shows that the wins did outweigh the losses, but not by a huge margin.

* **Max Drawdown (20.2%):** A 20% drop in our portfolio value is significant. This happened during a period in the market (Exit Dec 2022) that was choppy and sideways where the price frequently crisscrossed the 50-day moving average, generating a losing "whipsaw" trade.

* **Sharpe Ratio (0.99):** This is a measure of risk-adjusted return. A Sharpe ratio near 1.0 is generally considered pretty good. It suggests that while the total return was lower, we got a decent return for the amount of volatility (risk) we took on.

### Hypothesis

**The Signal is Too Simplistic**

Using just the price crossing a single moving average generates many false signals in sideways markets. Instead we can evolve to using two moving average signals, a short window and a long window. This was already implemented by Henry.

In [6]:
# Calculate the SMA
data['SMA100'] = data['Close'].rolling(window=100).mean()

# Determine the position state (1 for long, 0 for flat)
data['Position'] = np.where(data['SMA50'] > data['SMA100'], 1, 0)

# Calculate the signal (1 for buy, -1 for sell, 0 for hold)
data['Signal'] = data['Position'].diff()

# Display the rows where a trade signal occurred
print("Trade Signals (1 = Buy, -1 = Sell):")
print(data[data['Signal'] != 0].head())

Trade Signals (1 = Buy, -1 = Sell):
Price            Close        High         Low        Open    Volume  \
Date                                                                   
2022-01-03  453.210388  453.343222  449.548364  451.872697  72668200   
2022-09-09  388.617157  389.486914  384.660244  384.927859  76706900   
2022-10-17  352.036713  353.149967  342.881178  349.339961  93168200   
2022-12-30  368.702942  368.847552  364.846519  366.977214  84022200   
2023-10-17  425.035156  427.101771  421.555119  421.906035  75324700   

Price            SMA50  Position  Signal      SMA100  
Date                                                  
2022-01-03         NaN         0     NaN         NaN  
2022-09-09  384.386743         1     1.0  383.801823  
2022-10-17  374.883465         0    -1.0  375.119958  
2022-12-30  373.699642         1     1.0  373.510958  
2023-10-17  427.054236         0    -1.0  427.214884  


In [7]:
entries = data['Signal'] == 1
exits = data['Signal'] == -1

portfolio_dual_sma = vbt.Portfolio.from_signals(
    close=data['Close'],
    entries=entries,
    exits=exits,
    init_cash=100_000,
    freq='D'
)

# Print the performance statistics
print("\n--- Dual SMA (50/100) Backtest Performance ---")
print(portfolio_dual_sma.stats())


--- Dual SMA (50/100) Backtest Performance ---
Start                         2022-01-03 00:00:00
End                           2025-10-24 00:00:00
Period                          957 days 00:00:00
Start Value                              100000.0
End Value                           143045.470484
Total Return [%]                         43.04547
Benchmark Return [%]                    49.433909
Max Gross Exposure [%]                      100.0
Total Fees Paid                               0.0
Max Drawdown [%]                        12.883016
Max Drawdown Duration           181 days 00:00:00
Total Trades                                    4
Total Closed Trades                             3
Total Open Trades                               1
Open Trade PnL                       15238.663648
Win Rate [%]                            66.666667
Best Trade [%]                          22.388256
Worst Trade [%]                         -9.412977
Avg Winning Trade [%]                   18.833369
Av

**The dual SMA crossover is a significant improvement over the single SMA strategy.**

The results confirm our hypothesis:

* Total Return increased from 37.35% to 43.05%.

* Sharpe Ratio improved from 0.99 to 1.08, indicating better risk-adjusted returns.

* Most impressively, the Max Drawdown was nearly cut in half, from 20.26% to 12.88%. This is a huge improvement in risk management.

* The Win Rate jumped from 26% to 67%.

In [8]:
windows = np.arange(2, 201, 2)
fast_ma, slow_ma = vbt.MA.run_combs(data['Close'], window=windows, r=2, short_names=['fast', 'slow'])
entries = fast_ma.ma_crossed_above(slow_ma)
exits = fast_ma.ma_crossed_below(slow_ma)

combination_testing_portfolio = vbt.Portfolio.from_signals(
    close=data['Close'],
    entries=entries,
    exits=exits,
    init_cash=100_000,
    freq='D'
)

sharpe_ratios = combination_testing_portfolio.sharpe_ratio()
best_params = sharpe_ratios.idxmax()
best_fast_window, best_slow_window = best_params

print(f"\nBest Parameters Found:")
print(f"  Fast Window: {best_fast_window}")
print(f"  Slow Window: {best_slow_window}")
print(f"  Resulting Sharpe Ratio: {sharpe_ratios.max():.2f}")

sharpe_ratios.vbt.heatmap(
    xaxis_title='Slow Window',
    yaxis_title='Fast Window',
    title='In-Sample Sharpe Ratios for Dual SMA Strategy'
).show()


Best Parameters Found:
  Fast Window: 20
  Slow Window: 200
  Resulting Sharpe Ratio: 1.71


d:\Development\TigerQuant\demos\.venv\Lib\site-packages\jupyter_client\session.py:721: UserWarning:

Message serialization failed with:
Out of range float values are not JSON compliant: nan
Supporting this message is deprecated in jupyter-client 7, please make sure your message is JSON-compliant



In [9]:
def dual_sma_strategy(close_price, fast_window=50, slow_window=100):
    fast_sma = close_price.rolling(window=fast_window).mean()
    slow_sma = close_price.rolling(window=slow_window).mean()
    entries = (fast_sma > slow_sma) & (fast_sma.shift(1) <= slow_sma.shift(1))
    exits = (fast_sma < slow_sma) & (fast_sma.shift(1) >= slow_sma.shift(1))
    return entries, exits

def single_sma_strategy(close_price, window=50):
    sma = close_price.rolling(window=window).mean()
    entries = (close_price > sma) & (close_price.shift(1) <= sma.shift(1))
    exits = (close_price < sma) & (close_price.shift(1) >= sma.shift(1))
    return entries, exits

In [10]:
entries, exits = dual_sma_strategy(data['Close'], 50, 100)

test = vbt.Portfolio.from_signals(
    close=data['Close'],
    entries=entries,
    exits=exits,
    init_cash=100_000,
    freq='D'
)
test.stats()

Start                         2022-01-03 00:00:00
End                           2025-10-24 00:00:00
Period                          957 days 00:00:00
Start Value                              100000.0
End Value                           143045.470484
Total Return [%]                         43.04547
Benchmark Return [%]                    49.433909
Max Gross Exposure [%]                      100.0
Total Fees Paid                               0.0
Max Drawdown [%]                        12.883016
Max Drawdown Duration           181 days 00:00:00
Total Trades                                    4
Total Closed Trades                             3
Total Open Trades                               1
Open Trade PnL                       15238.663648
Win Rate [%]                            66.666667
Best Trade [%]                          22.388256
Worst Trade [%]                         -9.412977
Avg Winning Trade [%]                   18.833369
Avg Losing Trade [%]                    -9.412977


"The goal of this analysis is to prevent **overfitting**. We tune the strategy on one historical period (in-sample) and then validate it on a completely separate, unseen period (out-of-sample) to see if the performance is real or just a fluke.

In [29]:
in_sample = data[data.index < "2025-01-01"]
out_sample = data[data.index >= "2025-01-01"]

single_sma_entries, single_sma_exits = single_sma_strategy(data["Close"], window=50)
dual_sma_50_100_entries, dual_sma_50_100_exits = dual_sma_strategy(
    data["Close"], fast_window=50, slow_window=100
)
dual_sma_20_200_entries, dual_sma_20_200_exits = dual_sma_strategy(
    data["Close"], fast_window=20, slow_window=200
)

all_entries = pd.concat(
    {
        "Single_SMA_50": single_sma_entries,
        "SMA_50_100": dual_sma_50_100_entries,
        "SMA_20_200": dual_sma_20_200_entries,
    },
    axis=1,
)  # axis=1 means to stack horizontally (add more columns)

all_exits = pd.concat(
    {
        "Single_SMA_50": single_sma_exits,
        "SMA_50_100": dual_sma_50_100_exits,
        "SMA_20_200": dual_sma_20_200_exits,
    },
    axis=1,
)

in_entries = all_entries.loc[in_sample.index]
in_exits = all_exits.loc[in_sample.index]
out_entries = all_entries.loc[out_sample.index]
out_exits = all_exits.loc[out_sample.index]


print("--IN-SAMPLE PERFORMANCE--")
in_sample_portfolio = vbt.Portfolio.from_signals(
    close=in_sample["Close"],
    entries=in_entries,
    exits=in_exits,
    init_cash=100_000,
    freq="D",
)

display(
    in_sample_portfolio.stats(agg_func=None)
)  # Use agg_func=None to stop vectorbt from taking the average of all strategies

--IN-SAMPLE PERFORMANCE--


,Start,End,Period,Start Value,End Value,Total Return [%],Benchmark Return [%],Max Gross Exposure [%],Total Fees Paid,Max Drawdown [%],...,Avg Winning Trade [%],Avg Losing Trade [%],Avg Winning Trade Duration,Avg Losing Trade Duration,Profit Factor,Expectancy,Sharpe Ratio,Calmar Ratio,Omega Ratio,Sortino Ratio
Single_SMA_50,2022-01-03,2024-12-31,753 days,100000.0,113321.839605,13.321840,28.194135,100.0,0.0,20.259779,...,5.873103,-1.452769,55 days 06:51:25.714285714,5 days 02:39:59.999999999,1.528009,532.873584,0.523040,0.308473,1.097636,0.720778
SMA_50_100,2022-01-03,2024-12-31,753 days,100000.0,132077.312040,32.077312,28.194135,100.0,0.0,12.883016,...,15.278482,-9.412977,199 days 00:00:00,26 days 00:00:00,1.470345,2213.672608,1.049998,1.120670,1.202537,1.510357
SMA_20_200,2022-01-03,2024-12-31,753 days,100000.0,148545.575515,48.545576,28.194135,100.0,0.0,9.974318,...,NaN,NaN,NaT,NaT,NaN,NaN,1.626627,2.119935,1.322949,2.410217


As expected, on the data we used for tuning, our tuned `SMA_20_200` strategy looks like a superstar. It has the highest Sharpe Ratio. This is its home turf. 

Now for the real test. We take our winning parameters and run them on the 2025 data. This is the moment of truth.

In [30]:
print("\n--- OUT-OF-SAMPLE PERFORMANCE (Unseen validation data) ---")
out_sample_portfolio = vbt.Portfolio.from_signals(
    close=out_sample["Close"],
    entries=out_entries,
    exits=out_exits,
    init_cash=100_000,
    freq="D",
)

display(out_sample_portfolio.stats(agg_func=None))


--- OUT-OF-SAMPLE PERFORMANCE (Unseen validation data) ---


,Start,End,Period,Start Value,End Value,Total Return [%],Benchmark Return [%],Max Gross Exposure [%],Total Fees Paid,Max Drawdown [%],...,Avg Winning Trade [%],Avg Losing Trade [%],Avg Winning Trade Duration,Avg Losing Trade Duration,Profit Factor,Expectancy,Sharpe Ratio,Calmar Ratio,Omega Ratio,Sortino Ratio
Single_SMA_50,2025-01-02,2025-10-24,204 days,100000.0,121205.393982,21.205394,16.855547,100.0,0.0,2.984648,...,NaN,-0.311388,NaT,13 days,0.0,-311.214497,2.951451,13.760863,1.633236,4.756925
SMA_50_100,2025-01-02,2025-10-24,204 days,100000.0,111923.201921,11.923202,16.855547,100.0,0.0,2.984648,...,NaN,NaN,NaT,NaT,NaN,NaN,2.602389,7.481211,1.775951,3.981312
SMA_20_200,2025-01-02,2025-10-24,204 days,100000.0,117617.608450,17.617608,16.855547,100.0,0.0,2.984648,...,NaN,NaN,NaT,NaT,NaN,NaN,3.232812,11.286865,1.892322,5.268916


How much did the performance **degrade**?

`SMA_20_200`'s return dropped significantly from 48% toi 17%, however, the length of in_sample was WAY longer than the out of sample time period. It still outperformed the benchmark return of buying and holding SPY for the entire period. However, it dropped behind our OG single SMA strategy. 

One thing that should be noted is the sharpe ratio of our tuned `SMA_20_200' strategy. All of our sharpe's are quite high, but 3.2 is ridiculous, especially considered the strategy also outpreformed our benchmark. This means our trading strategy was getting the same/better return for much less risk.

The max drawdown in each of our startegies was the same. Interesting. However during our in sample testing, our tuned strategy showed that it limited drawdown way better than the pther strategies. 

In [31]:
out_sample_portfolio['SMA_20_200'].plot().show()

Some questions we should look for when analyzing a strategy more deeply are: 

* **Drawdowns**: When did the biggest drop in portfolio value happen? Was the market trending down, or was it choppy and moving sideways?

* **Missed Opportunities**: Were there strong uptrends that your strategy was too slow to enter? Why?

* **Losing Trades:** Look at the losing trades. What did the market look like? Was it a "whipsaw" (price quickly reversed after you entered)?

In [40]:
records = out_sample_portfolio.get_trades().records_readable

display(records[records['Column'] == 'SMA_20_200'])

,Exit Trade Id,Column,Size,Entry Timestamp,Avg Entry Price,Entry Fees,Exit Timestamp,Avg Exit Price,Exit Fees,PnL,Return,Direction,Status,Position Id
4,4,SMA_20_200,173.669411,2025-05-23,575.806641,0.0,2025-10-24,677.25,0.0,17617.60845,0.176176,Long,Open,4


Literally just one trade lol.

What would've been the best window combination on the out of sample data and how did it perform comapared to itself in sample?

In [43]:
windows = np.arange(2, 201, 2)
fast_ma, slow_ma = vbt.MA.run_combs(out_sample['Close'], window=windows, r=2, short_names=['fast', 'slow'])
entries = fast_ma.ma_crossed_above(slow_ma)
exits = fast_ma.ma_crossed_below(slow_ma)

combination_testing_portfolio = vbt.Portfolio.from_signals(
    close=out_sample['Close'],
    entries=entries,
    exits=exits,
    init_cash=100_000,
    freq='D'
)

sharpe_ratios = combination_testing_portfolio.sharpe_ratio()
best_params = sharpe_ratios.idxmax()
best_fast_window, best_slow_window = best_params

print(f"\nBest Parameters Found:")
print(f"  Fast Window: {best_fast_window}")
print(f"  Slow Window: {best_slow_window}")
print(f"  Resulting Sharpe Ratio: {sharpe_ratios.max():.2f}")

sharpe_ratios.vbt.heatmap(
    xaxis_title='Slow Window',
    yaxis_title='Fast Window',
    title='Out of Sample Sharpe Ratios for Dual SMA Strategy'
).show()


Best Parameters Found:
  Fast Window: 2
  Slow Window: 90
  Resulting Sharpe Ratio: inf


d:\Development\TigerQuant\demos\.venv\Lib\site-packages\jupyter_client\session.py:721: UserWarning:

Message serialization failed with:
Out of range float values are not JSON compliant: nan
Supporting this message is deprecated in jupyter-client 7, please make sure your message is JSON-compliant



And for just single SMA on out of sample?

In [ ]:
windows = np.arange(2, 201, 2)
sma = vbt.MA.run(out_sample['Close'], window=windows, short_name='sma')
price = out_sample['Close'].vbt.tile(len(windows))
price.columns = sma.ma.columns
entries = price > sma.ma
exits = price < sma.ma

portfolio = vbt.Portfolio.from_signals(
    close=out_sample['Close'],
    entries=entries,
    exits=exits,
    init_cash=100_000,
    freq='D'
)

sharpe_ratios = portfolio.sharpe_ratio()
best_param = sharpe_ratios.idxmax()
best_window = best_param

print(f"\nBest Parameter Found:")
print(f"  Window: {best_window}")
print(f"  Resulting Sharpe Ratio: {sharpe_ratios.max():.2f}")

sharpe_ratios.vbt.heatmap(
    title='Out of Sample Sharpe Ratios for Single SMA Strategy'
).show()


Best Parameter Found:
  Window: 44
  Resulting Sharpe Ratio: 3.46


Single SMA in sample?

In [59]:
windows = np.arange(2, 201, 2)
sma = vbt.MA.run(in_sample['Close'], window=windows, short_name='sma')
price = in_sample['Close'].vbt.tile(len(windows))
price.columns = sma.ma.columns
entries = price > sma.ma
exits = price < sma.ma

portfolio = vbt.Portfolio.from_signals(
    close=in_sample['Close'],
    entries=entries,
    exits=exits,
    init_cash=100_000,
    freq='D'
)

sharpe_ratios = portfolio.sharpe_ratio()
best_param = sharpe_ratios.idxmax()
best_window = best_param

print(f"\nBest Parameter Found:")
print(f"  Window: {best_window}")
print(f"  Resulting Sharpe Ratio: {sharpe_ratios.max():.2f}")

sharpe_ratios.vbt.heatmap(
    title='In Sample Sharpe Ratios for Single SMA Strategy'
).show()


Best Parameter Found:
  Window: 200
  Resulting Sharpe Ratio: 1.40


Let's test all strategies against each other now and print the stats.

In [61]:
in_sample = data[data.index < "2025-01-01"]
out_sample = data[data.index >= "2025-01-01"]

sma_50_entries, sma_50_exits = single_sma_strategy(data["Close"], window=50)
sma_44_entries, sma_44_exits = single_sma_strategy(data["Close"], window=44)
sma_200_entries, sma_200_exits = single_sma_strategy(data["Close"], window=200)

sma_50_100_entries, sma_50_100_exits = dual_sma_strategy(
    data["Close"], fast_window=50, slow_window=100
)
sma_20_200_entries, sma_20_200_exits = dual_sma_strategy(
    data["Close"], fast_window=20, slow_window=200
)
sma_2_90_entries, sma_2_90_exits = dual_sma_strategy(
    data["Close"], fast_window=2, slow_window=90
)

all_entries = pd.concat(
    {
        "Single_SMA_50": sma_50_entries,
        "Single_SMA_44": sma_44_entries,
        "Single_SMA_200": sma_200_entries,
        "SMA_50_100": sma_50_100_entries,
        "SMA_20_200": sma_20_200_entries,
        "SMA_2_90": sma_2_90_entries,
    },
    axis=1,
)  # axis=1 means to stack horizontally (add more columns)

all_exits = pd.concat(
    {
        "Single_SMA_50": sma_50_exits,
        "Single_SMA_44": sma_44_exits,
        "Single_SMA_200": sma_200_exits,
        "SMA_50_100": sma_50_100_exits,
        "SMA_20_200": sma_20_200_exits,
        "SMA_2_90": sma_2_90_exits,
    },
    axis=1,
)

in_entries = all_entries.loc[in_sample.index]
in_exits = all_exits.loc[in_sample.index]
out_entries = all_entries.loc[out_sample.index]
out_exits = all_exits.loc[out_sample.index]


print("--IN-SAMPLE PERFORMANCE--")
in_sample_portfolio = vbt.Portfolio.from_signals(
    close=in_sample["Close"],
    entries=in_entries,
    exits=in_exits,
    init_cash=100_000,
    freq="D",
)

display(
    in_sample_portfolio.stats(agg_func=None)[['Total Return [%]', 'Benchmark Return [%]', 'Max Drawdown [%]', 'Sharpe Ratio']]
)  # Use agg_func=None to stop vectorbt from taking the average of all strategies

print("\n--- OUT-OF-SAMPLE PERFORMANCE (Unseen validation data) ---")
out_sample_portfolio = vbt.Portfolio.from_signals(
    close=out_sample["Close"],
    entries=out_entries,
    exits=out_exits,
    init_cash=100_000,
    freq="D",
)

display(out_sample_portfolio.stats(agg_func=None)[['Total Return [%]', 'Benchmark Return [%]', 'Max Drawdown [%]', 'Sharpe Ratio']])

--IN-SAMPLE PERFORMANCE--


,Total Return [%],Benchmark Return [%],Max Drawdown [%],Sharpe Ratio
Single_SMA_50,13.321840,28.194135,20.259779,0.523040
Single_SMA_44,19.598786,28.194135,17.717990,0.727255
Single_SMA_200,40.347432,28.194135,9.061764,1.403405
SMA_50_100,32.077312,28.194135,12.883016,1.049998
SMA_20_200,48.545576,28.194135,9.974318,1.626627
SMA_2_90,29.931048,28.194135,13.744953,1.019005



--- OUT-OF-SAMPLE PERFORMANCE (Unseen validation data) ---


,Total Return [%],Benchmark Return [%],Max Drawdown [%],Sharpe Ratio
Single_SMA_50,21.205394,16.855547,2.984648,2.951451
Single_SMA_44,21.413379,16.855547,2.984648,2.974737
Single_SMA_200,15.717509,16.855547,2.984648,2.739386
SMA_50_100,11.923202,16.855547,2.984648,2.602389
SMA_20_200,17.617608,16.855547,2.984648,3.232812
SMA_2_90,17.408748,16.855547,5.183911,2.615170


**The best-performing strategy in-sample is rarely the best-performing strategy out-of-sample.** This is the perfect demonstration of overfitting.

Let's discuss...

1. **In-Sample: The "Perfect" Strategy Emerges.** During the training period (before 2025), the `SMA_20_200` strategy was the undisputed champion. It crushed the benchmark return (48% vs 28%), had the lowest drawdown (under 10%), and the highest Sharpe Ratio (1.63). If we stopped here, we would confidently declare it the best strategy. The other dual SMA strategies also beat the benchmark, confirming that Hypothesis A was a good one. The single SMA strategies were clear underperformers.

2. **Out-of-Sample: The Great Reversal.** This is the crucial part. When tested on unseen 2025 data, the story completely flipped:

    * Our in-sample champion, `SMA_20_200`, only barely beat the benchmark (17.6% vs 16.9%). While its Sharpe Ratio was high, its actual profit advantage was minimal.

    * The simple `Single_SMA_44` and `Single_SMA_50` strategies, which were the worst performers in-sample, suddenly became the best performers out-of-sample, beating the benchmark by a healthy margin (~21% vs 16.9%).

    * **The Lesson:** We have perfectly demonstrated **overfitting**. The `SMA_20_200` was so perfectly tuned to the specific market conditions of 2022-2024 that its predictive power failed on new data. The simpler, less "optimized" models proved to be more robust and adaptable.

In [63]:
display(out_sample_portfolio.stats(agg_func=None).T)

,Single_SMA_50,Single_SMA_44,Single_SMA_200,SMA_50_100,SMA_20_200,SMA_2_90
Start,2025-01-02 00:00:00,2025-01-02 00:00:00,2025-01-02 00:00:00,2025-01-02 00:00:00,2025-01-02 00:00:00,2025-01-02 00:00:00
End,2025-10-24 00:00:00,2025-10-24 00:00:00,2025-10-24 00:00:00,2025-10-24 00:00:00,2025-10-24 00:00:00,2025-10-24 00:00:00
Period,204 days 00:00:00,204 days 00:00:00,204 days 00:00:00,204 days 00:00:00,204 days 00:00:00,204 days 00:00:00
Start Value,100000.0,100000.0,100000.0,100000.0,100000.0,100000.0
End Value,121205.393982,121413.378888,115717.508893,111923.201921,117617.60845,117408.748302
Total Return [%],21.205394,21.413379,15.717509,11.923202,17.617608,17.408748
Benchmark Return [%],16.855547,16.855547,16.855547,16.855547,16.855547,16.855547
Max Gross Exposure [%],100.0,100.0,100.0,100.0,100.0,100.0
Total Fees Paid,0.0,0.0,0.0,0.0,0.0,0.0
Max Drawdown [%],2.984648,2.984648,2.984648,2.984648,2.984648,5.183911


For every strategy except `SMA_2_90`, the Avg winning trade [%] and avg winning trade duration are NaN and NaT respectively. Why is this? My guess is that these strategies don't have enough time to exit their positions in the short out of sample time frame, so the insane profit we see is just calculated from the and close position